In [0]:
# ============================================================
# Customer Landing Validation
# ============================================================

import json
from datetime import datetime, timezone



# ------------------------------------------------------------
# 1. Input parameter from ADF / Databricks Job
# ------------------------------------------------------------

dbutils.widgets.text(
    "customer_path",
    "abfss://landing@olistdev1.dfs.core.windows.net/SAP/Customer/olist_customers_dataset.csv"
)
#pipelinne run id param from adf
dbutils.widgets.text(
    "pipeline_run_id",
    ""
)


customer_path = dbutils.widgets.get("customer_path")
pipeline_run_id = dbutils.widgets.get("pipeline_run_id")
print(f"Processing file: {customer_path}")
print(f"ADF Pipeline Run ID: {pipeline_run_id}")

# ------------------------------------------------------------
# 2. Read Customer Landing File
# ------------------------------------------------------------

df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(customer_path)
)

print("Customer file successfully read.")


# ------------------------------------------------------------
# 3. Validate Record Count
# ------------------------------------------------------------

customer_count = df.count()

print(f"Customer record count: {customer_count}")


# ------------------------------------------------------------
# 4. Customer Data Validation
# ------------------------------------------------------------

from pyspark.sql.functions import col, trim, sum, when


validation_status = "PASS"
failed_validation = None
validation_error = None


try:

    # --------------------------------------------------------
    # 4.1 Validate Required Columns
    # --------------------------------------------------------

    required_columns = [
        "customer_id",
        "customer_unique_id",
        "customer_zip_code_prefix",
        "customer_city",
        "customer_state"
    ]

    actual_columns = df.columns

    missing_columns = [
        column
        for column in required_columns
        if column not in actual_columns
    ]
   
    if missing_columns:

        raise Exception(
            f"CUSTOMER_SCHEMA_VALIDATION: FAIL - "
            f"Missing columns: {missing_columns}"
        )

    print("CUSTOMER_SCHEMA_VALIDATION: PASS")


    # --------------------------------------------------------
    # 4.2 Validate Record Count
    # --------------------------------------------------------

    if customer_count == 0:

        raise Exception(
            "CUSTOMER_DATA_VALIDATION: FAIL - "
            "Customer file contains zero records"
        )

    print("CUSTOMER_DATA_VALIDATION: PASS")


    # --------------------------------------------------------
    # 4.3 Validate Null / Blank Values
    # --------------------------------------------------------

    null_counts = df.select(
        *[
            sum(
                when(
                    col(column).isNull() |
                    (trim(col(column).cast("string")) == ""),
                    1
                ).otherwise(0)
            ).alias(column)
            for column in required_columns
        ]
    ).collect()[0]

    null_results = {
        column: null_counts[column]
        for column in required_columns
    }

    invalid_null_columns = {
        column: count
        for column, count in null_results.items()
        if count > 0
    }

    if invalid_null_columns:

        raise Exception(
            f"CUSTOMER_NULL_VALIDATION: FAIL - "
            f"Null or blank values found: "
            f"{invalid_null_columns}"
        )

    print("CUSTOMER_NULL_VALIDATION: PASS")


    # --------------------------------------------------------
    # 4.4 Validate Duplicate Customer IDs
    # --------------------------------------------------------

    duplicate_customer_ids = (
        df.groupBy("customer_id")
          .count()
          .filter(col("count") > 1)
    )

    duplicate_count = duplicate_customer_ids.count()

    if duplicate_count > 0:

        raise Exception(
            f"CUSTOMER_DUPLICATE_VALIDATION: FAIL - "
            f"{duplicate_count} duplicate customer_id values found"
        )

    print("CUSTOMER_DUPLICATE_VALIDATION: PASS")


    # --------------------------------------------------------
    # 4.5 Validate Customer State
    # --------------------------------------------------------

    valid_states = {
        "AC", "AL", "AP", "AM", "BA", "CE", "DF",
        "ES", "GO", "MA", "MT", "MS", "MG", "PA",
        "PB", "PR", "PE", "PI", "RJ", "RN", "RS",
        "RO", "RR", "SC", "SP", "SE", "TO"
    }

    invalid_states = (
        df.select("customer_state")
          .withColumn(
              "customer_state",
              trim(col("customer_state"))
          )
          .filter(
              ~col("customer_state").isin(valid_states)
          )
          .groupBy("customer_state")
          .count()
    )

    invalid_state_count = invalid_states.count()

    if invalid_state_count > 0:

        invalid_state_values = [
            row["customer_state"]
            for row in (
                invalid_states
                .select("customer_state")
                .distinct()
                .collect()
            )
        ]

        raise Exception(
            f"CUSTOMER_STATE_VALIDATION: FAIL - "
            f"Invalid customer_state values found: "
            f"{invalid_state_values}"
        )

    print("CUSTOMER_STATE_VALIDATION: PASS")


    # --------------------------------------------------------
    # 4.6 Validate Customer ZIP Code Prefix
    # --------------------------------------------------------

    zip_values = (
        df.select("customer_zip_code_prefix")
          .withColumn(
              "zip_code",
              trim(col("customer_zip_code_prefix").cast("string"))
          )
    )

    invalid_zip_codes = (
        zip_values
        .filter(
            ~col("zip_code").rlike("^[0-9]{4,5}$")
        )
    )

    invalid_zip_count = invalid_zip_codes.count()

    if invalid_zip_count > 0:

        invalid_zip_values = [
            row["zip_code"]
            for row in (
                invalid_zip_codes
                .select("zip_code")
                .distinct()
                .collect()
            )
        ]

        raise Exception(
            f"CUSTOMER_ZIP_VALIDATION: FAIL - "
            f"Invalid customer_zip_code_prefix values found: "
            f"{invalid_zip_values}"
        )

    print("CUSTOMER_ZIP_VALIDATION: PASS")


    print("CUSTOMER_VALIDATION: ALL CHECKS PASSED")


except Exception as e:

    validation_status = "FAIL"
    validation_error = str(e)

    if "CUSTOMER_SCHEMA_VALIDATION" in validation_error:
        failed_validation = "CUSTOMER_SCHEMA_VALIDATION"

    elif "CUSTOMER_DATA_VALIDATION" in validation_error:
        failed_validation = "CUSTOMER_DATA_VALIDATION"

    elif "CUSTOMER_NULL_VALIDATION" in validation_error:
        failed_validation = "CUSTOMER_NULL_VALIDATION"

    elif "CUSTOMER_DUPLICATE_VALIDATION" in validation_error:
        failed_validation = "CUSTOMER_DUPLICATE_VALIDATION"

    elif "CUSTOMER_STATE_VALIDATION" in validation_error:
        failed_validation = "CUSTOMER_STATE_VALIDATION"

    elif "CUSTOMER_ZIP_VALIDATION" in validation_error:
        failed_validation = "CUSTOMER_ZIP_VALIDATION"

    else:
        failed_validation = "UNKNOWN_VALIDATION_ERROR"

    print(failed_validation + ": FAIL")
    print(validation_error)

audit_path = (
    "abfss://landing@olistdev1.dfs.core.windows.net/"
    "Audit/SAP/Customer/customer_validation_result.json"
)

# ------------------------------------------------------------
# 5. Audit Result
# ------------------------------------------------------------
import json
import uuid
from datetime import datetime, timezone


# ------------------------------------------------------------
# 5. Audit Result
# ------------------------------------------------------------

# Generate unique execution ID
run_id = pipeline_run_id

# Execution timestamp
validation_timestamp = datetime.now(timezone.utc).isoformat()

# Audit metadata
source_system = "SAP"
entity = "Customer"

# Build audit result
result = {
    "run_id": run_id,
    "source_system": source_system,
    "entity": entity,
    "status": validation_status,
    "failed_validation": failed_validation,
    "error_message": validation_error,
    "record_count": customer_count,
    "source_file": customer_path,
    "validation_timestamp": validation_timestamp
}

result_json = json.dumps(result)

# ------------------------------------------------------------
# 6. Write Audit Result to ADLS
# ------------------------------------------------------------

audit_directory = (
    "abfss://landing@olistdev1.dfs.core.windows.net/"
    "Audit/SAP/Customer/"
)

audit_path = (
    audit_directory +
    f"customer_validation_{run_id}.json"
)

dbutils.fs.put(
    audit_path,
    result_json,
    overwrite=False
)

print(f"Validation result written to: {audit_path}")


# ------------------------------------------------------------
# 7. Display Sample Data
# ------------------------------------------------------------

display(df.limit(5))


# ------------------------------------------------------------
# 8. Return Result
# ------------------------------------------------------------

if validation_status == "FAIL":

    raise Exception(
        f"{failed_validation}: {validation_error}"
    )

dbutils.notebook.exit(result_json)

In [0]:
# import json

# result = {
#     "status": "PASS",
#     "record_count": customer_count,
#     "source_file": customer_path
# }

# dbutils.notebook.exit(json.dumps(result))

negative testing


we create a new file with bad data

In [0]:
old_audit_path = (
    "abfss://landing@olistdev1.dfs.core.windows.net/"
    "Audit/SAP/Customer/customer_validation_result.json"
)

dbutils.fs.rm(old_audit_path, True)

print("Old audit file removed.")

In [0]:
%sql
SELECT *
FROM read_files(
    'abfss://landing@olistdev1.dfs.core.windows.net/Audit/SAP/Customer/',
    format => 'json'
)
ORDER BY validation_timestamp DESC;

In [0]:
%sql
SELECT *
FROM read_files(
  'abfss://landing@olistdev1.dfs.core.windows.net/Audit/SAP/Customer/',
  format => 'json'
)
ORDER BY validation_timestamp DESC;